In [3]:
import concurrent.futures
from datetime import datetime, timezone
from urllib.parse import parse_qs, urlparse
import pandas as pd
import requests


def es_link_valido(url: str, timeout: int = 5) -> bool:
    """Verifica si una URL es estructuralmente válida, no ha expirado

    y responde con un código HTTP 200 OK.
    """
    if (
        pd.isna(url)
        or not isinstance(url, str)
        or not url.startswith(("http://", "https://"))
    ):
        return False

    # 1. Validación offline rápida para firmas de Google Cloud Storage (X-Goog-Expires)
    try:
        parsed = urlparse(url)
        params = parse_qs(parsed.query)
        if "X-Goog-Date" in params and "X-Goog-Expires" in params:
            date_str = params["X-Goog-Date"][0]
            expires_sec = int(params["X-Goog-Expires"][0])
            created_at = datetime.strptime(
                date_str, "%Y%m%dT%H%M%SZ"
            ).replace(tzinfo=timezone.utc)
            # Si el token de la URL ya sobrepasó su tiempo de vida útil
            if datetime.now(timezone.utc).timestamp() > (
                created_at.timestamp() + expires_sec
            ):
                return False
    except Exception:
        pass

    # 2. Validación de disponibilidad y descarga vía HTTP
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    try:
        # Intento mediante petición HEAD (rápida, solo consulta cabeceras)
        response = requests.head(
            url, timeout=timeout, allow_redirects=True, headers=headers
        )
        if response.status_code == 200:
            return True
        elif response.status_code in (401, 403, 404, 410):
            return False

        # Si el servidor no soporta HEAD (405 Method Not Allowed), intenta un GET ligero
        response = requests.get(
            url, timeout=timeout, stream=True, headers=headers
        )
        return response.status_code == 200
    except Exception:
        return False


def filtrar_csv_por_links(
    archivo_entrada: str = "data.csv",
    archivo_salida: str = "data_filtrado.csv",
    columna_link: str = "link_imagen",
    max_trabajadores: int = 30,
) -> pd.DataFrame:
    # Cargar CSV original
    df = pd.read_csv(archivo_entrada)

    urls = df[columna_link].tolist()
    print(
        f"Validando {len(urls)} enlaces de la columna '{columna_link}' en paralelo..."
    )

    # Procesar verificación multihilo para mayor rapidez
    with concurrent.futures.ThreadPoolExecutor(
        max_workers=max_trabajadores
    ) as executor:
        mascara_validos = list(
            executor.map(lambda u: es_link_valido(u), urls)
        )

    # Filtrar manteniendo el esquema y los tipos de datos originales
    df_filtrado = df[mascara_validos].copy()

    print(f"Registros originales: {len(df)}")
    print(f"Registros válidos/descargables: {len(df_filtrado)}")
    print(
        f"Registros descartados (vencidos/inválidos): {len(df) - len(df_filtrado)}"
    )

    # Guardar manteniendo exactamente la misma estructura de columnas
    df_filtrado.to_csv(archivo_salida, index=False)
    return df_filtrado


if __name__ == "__main__":
    # Ejecución principal
    df_resultado = filtrar_csv_por_links(
        archivo_entrada="data.csv",
        archivo_salida="data_filtrado.csv",
        columna_link="link_imagen",
        max_trabajadores=30,  # Ajustar concurrencia según ancho de banda
    )


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\luigu\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\luigu\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\luigu\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\luigu\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import